# ZBUE: Milestone 1 — Transaction Intelligence & Feature Store

> **Zeyro Behavioral Underwriting Engine**  
> Dataset: `Customer_financial_profiles.csv` (20,000 rows · 4,941 clients · 500 merchants)  
> Goal: Construct the Feature Store organized by Feature Families for training the Income Estimation Model.

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Environment ready.")

Environment ready.


## 1. Load Raw Data

In [2]:
df_raw = pd.read_csv('../retail_lending/data/raw/Customer_financial_profiles.csv')
df_raw['date'] = pd.to_datetime(df_raw['date'])
df_raw['monthly_income'] = df_raw['yearly_income'] / 12.0
print(f"Shape: {df_raw.shape}")
display(df_raw.head(3))

Shape: (20000, 22)


,id,current_age,birth_year,birth_month,gender,address,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,transaction_id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,monthly_income
0,1,64,1961,3,Female,"Street 286, Indore, Madhya Pradesh - 452645",86125,1033501,280435,715,5,1251,2024-03-28,CLTIND00104,CRD001042,1483.94,Yes,MCH00496,Mumbai,Maharashtra,452645,86125.08
1,2,39,1986,12,Female,"Street 268, Kolkata, West Bengal - 700555",68414,820965,0,834,3,9976,2025-09-20,CLTIND00213,CRD002132,1274.72,Yes,MCH00374,Mumbai,Maharashtra,700555,68413.75
2,3,50,1975,7,Female,"Street 262, Lucknow, Uttar Pradesh - 226797",22595,271138,0,609,2,6084,2024-10-02,CLTIND03292,CRD032922,7622.04,No,MCH00108,Kolkata,West Bengal,226797,22594.83


## 2. Dataset Quality Checks

In [3]:
print("=== Missing Values ===")
print(df_raw.isnull().sum().to_string())
print()
print(f"Unique clients   : {df_raw['client_id'].nunique():,}")
print(f"Unique merchants : {df_raw['merchant_id'].nunique():,}")
print(f"Date range       : {df_raw['date'].min().date()} → {df_raw['date'].max().date()}")
print()
print("=== Income Distribution ===")
display(df_raw['yearly_income'].describe())

=== Missing Values ===
id                   0
current_age          0
birth_year           0
birth_month          0
gender               0
address              0
per_capita_income    0
yearly_income        0
total_debt           0
credit_score         0
num_credit_cards     0
transaction_id       0
date                 0
client_id            0
card_id              0
amount               0
use_chip             0
merchant_id          0
merchant_city        0
merchant_state       0
zip                  0
monthly_income       0

Unique clients   : 4,941
Unique merchants : 500
Date range       : 2023-01-01 → 2025-10-31

=== Income Distribution ===


count     20000.00
mean     734813.05
std      424251.19
min      150000.00
25%      428047.00
50%      658756.00
75%      898109.00
max     3000000.00
Name: yearly_income, dtype: float64

## 3. Feature Family Construction
Each feature family is computed **per client**, then merged into the Feature Store.

In [4]:
# --- INCOME FEATURES ---
def compute_income_features(df):
    grp = df.groupby('client_id')
    
    # Monthly income from yearly
    inc = df[['client_id','monthly_income','credit_score','yearly_income',
               'total_debt','num_credit_cards','current_age']].drop_duplicates('client_id').set_index('client_id')
    
    # Transaction frequency as proxy for salary periodicity
    tx_per_month = grp['transaction_id'].count() / df.groupby('client_id')['date'].apply(
        lambda x: max((x.max() - x.min()).days / 30, 1))
    
    # Income growth (per-capita income vs overall mean)
    income_per_cap = df.drop_duplicates('client_id').set_index('client_id')['per_capita_income']
    
    feat = pd.DataFrame({
        'monthly_income'           : inc['monthly_income'],
        'yearly_income'            : inc['yearly_income'],
        'income_per_capita'        : income_per_cap,
        'credit_score'             : inc['credit_score'],
        'total_debt'               : inc['total_debt'],
        'num_credit_cards'         : inc['num_credit_cards'],
        'age'                      : inc['current_age'],
        'tx_frequency_per_month'   : tx_per_month,
    })
    return feat

income_feats = compute_income_features(df_raw)
print(f"Income Features: {income_feats.shape}")
display(income_feats.head(3))

Income Features: (4941, 8)


,monthly_income,yearly_income,income_per_capita,credit_score,total_debt,num_credit_cards,age,tx_frequency_per_month
client_id,,,,,,,,
CLTIND00001,25333.00,303996,25333,728,0,1,32,0.20
CLTIND00002,59420.75,713049,59421,825,315841,5,19,0.56
CLTIND00004,36796.58,441559,36797,757,121659,1,56,1.00


In [5]:
# --- SPENDING FEATURES ---
def compute_spending_features(df):
    grp = df.groupby('client_id')
    
    total_spend = grp['amount'].sum()
    tx_count    = grp['amount'].count()
    
    feat = pd.DataFrame({
        'total_spend'         : total_spend,
        'avg_tx_amount'       : grp['amount'].mean(),
        'max_tx_amount'       : grp['amount'].max(),
        'min_tx_amount'       : grp['amount'].min(),
        'tx_count'            : tx_count,
        'chip_usage_ratio'    : df.assign(chip=df['use_chip']=='Yes').groupby('client_id')['chip'].mean(),
        'merchant_diversity'  : grp['merchant_id'].nunique(),
        'city_diversity'      : grp['merchant_city'].nunique(),
    })
    return feat

spending_feats = compute_spending_features(df_raw)
print(f"Spending Features: {spending_feats.shape}")
display(spending_feats.head(3))

Spending Features: (4941, 8)


,total_spend,avg_tx_amount,max_tx_amount,min_tx_amount,tx_count,chip_usage_ratio,merchant_diversity,city_diversity
client_id,,,,,,,,
CLTIND00001,9593.16,2398.29,3098.96,1650.38,4,0.25,4,4
CLTIND00002,45021.67,2813.85,5300.50,1119.11,16,0.62,16,11
CLTIND00004,4319.10,4319.10,4319.10,4319.10,1,1.00,1,1


In [6]:
# --- BEHAVIOUR FEATURES ---
def compute_behaviour_features(df):
    grp = df.groupby('client_id')
    
    # Spending entropy (higher = more diverse spending)
    def spending_entropy(amounts):
        p = amounts / amounts.sum()
        return -(p * np.log(p + 1e-9)).sum()
    
    entropy = grp['amount'].apply(spending_entropy)
    
    # Spending volatility (std/mean coefficient of variation)
    volatility = grp['amount'].std() / grp['amount'].mean()
    
    # Weekend spending ratio
    df['is_weekend'] = df['date'].dt.dayofweek >= 5
    weekend_ratio = df.groupby('client_id')['is_weekend'].mean()
    
    # Month-over-month spend std (stability)
    monthly_spend = df.groupby(['client_id', pd.Grouper(key='date', freq='ME')])['amount'].sum()
    mom_volatility = monthly_spend.groupby('client_id').std() / monthly_spend.groupby('client_id').mean()
    
    feat = pd.DataFrame({
        'spending_entropy'    : entropy,
        'spending_volatility' : volatility,
        'weekend_spend_ratio' : weekend_ratio,
        'mom_spend_volatility': mom_volatility,
    })
    return feat

behaviour_feats = compute_behaviour_features(df_raw)
print(f"Behaviour Features: {behaviour_feats.shape}")
display(behaviour_feats.head(3))

Behaviour Features: (4941, 4)


,spending_entropy,spending_volatility,weekend_spend_ratio,mom_spend_volatility
client_id,,,,
CLTIND00001,1.36,0.26,0.25,0.26
CLTIND00002,2.67,0.47,0.25,0.47
CLTIND00004,-0.00,NaN,0.00,NaN


In [7]:
# --- STABILITY FEATURES ---
def compute_stability_features(df):
    # Recency: days since last transaction
    last_tx = df.groupby('client_id')['date'].max()
    snapshot_date = df['date'].max()
    recency = (snapshot_date - last_tx).dt.days
    
    # Tenure: days from first to last transaction
    first_tx = df.groupby('client_id')['date'].min()
    tenure = (last_tx - first_tx).dt.days
    
    # Active months count
    active_months = df.groupby('client_id')['date'].apply(
        lambda x: x.dt.to_period('M').nunique())
    
    feat = pd.DataFrame({
        'recency_days'        : recency,
        'customer_tenure_days': tenure,
        'active_months'       : active_months,
    })
    return feat

stability_feats = compute_stability_features(df_raw)
print(f"Stability Features: {stability_feats.shape}")
display(stability_feats.head(3))

Stability Features: (4941, 3)


,recency_days,customer_tenure_days,active_months
client_id,,,
CLTIND00001,85,607,4
CLTIND00002,171,852,16
CLTIND00004,606,0,1


In [8]:
# --- AFFORDABILITY FEATURES ---
def compute_affordability_features(df):
    client_info = df.drop_duplicates('client_id').set_index('client_id')
    monthly_income = client_info['monthly_income']
    total_debt = client_info['total_debt']
    
    # DTI: total debt / yearly income
    dti = total_debt / client_info['yearly_income'].replace(0, np.nan)
    
    # Debt per card
    debt_per_card = total_debt / client_info['num_credit_cards'].replace(0, 1)
    
    # Monthly spend from transaction data
    monthly_spend = df.groupby('client_id')['amount'].sum() / df.groupby('client_id')['date'].apply(
        lambda x: max((x.max()-x.min()).days / 30, 1))
    
    disposable = monthly_income - monthly_spend
    foir = monthly_spend / monthly_income.replace(0, np.nan)
    savings_rate = disposable / monthly_income.replace(0, np.nan)
    
    feat = pd.DataFrame({
        'monthly_spend_est'  : monthly_spend,
        'disposable_income'  : disposable,
        'foir'               : foir.clip(0, 1),
        'savings_rate'       : savings_rate.clip(-1, 1),
        'dti'                : dti.clip(0, 5),
        'debt_per_card'      : debt_per_card,
    })
    return feat

affordability_feats = compute_affordability_features(df_raw)
print(f"Affordability Features: {affordability_feats.shape}")
display(affordability_feats.head(3))

Affordability Features: (4941, 6)


,monthly_spend_est,disposable_income,foir,savings_rate,dti,debt_per_card
client_id,,,,,,
CLTIND00001,474.13,24858.87,0.02,0.98,0.00,0.00
CLTIND00002,1585.27,57835.48,0.03,0.97,0.44,63168.20
CLTIND00004,4319.10,32477.48,0.12,0.88,0.28,121659.00


## 4. Merge Feature Families into Feature Store

In [9]:
feature_store = (
    income_feats
    .join(spending_feats, how='left')
    .join(behaviour_feats, how='left')
    .join(stability_feats, how='left')
    .join(affordability_feats, how='left')
    .reset_index()
)

print(f"Feature Store Shape: {feature_store.shape}")
print(f"Columns ({len(feature_store.columns)}): {list(feature_store.columns)}")
display(feature_store.head(3))

Feature Store Shape: (4941, 30)
Columns (30): ['client_id', 'monthly_income', 'yearly_income', 'income_per_capita', 'credit_score', 'total_debt', 'num_credit_cards', 'age', 'tx_frequency_per_month', 'total_spend', 'avg_tx_amount', 'max_tx_amount', 'min_tx_amount', 'tx_count', 'chip_usage_ratio', 'merchant_diversity', 'city_diversity', 'spending_entropy', 'spending_volatility', 'weekend_spend_ratio', 'mom_spend_volatility', 'recency_days', 'customer_tenure_days', 'active_months', 'monthly_spend_est', 'disposable_income', 'foir', 'savings_rate', 'dti', 'debt_per_card']


,client_id,monthly_income,yearly_income,income_per_capita,credit_score,total_debt,num_credit_cards,age,tx_frequency_per_month,total_spend,avg_tx_amount,max_tx_amount,min_tx_amount,tx_count,chip_usage_ratio,merchant_diversity,city_diversity,spending_entropy,spending_volatility,weekend_spend_ratio,mom_spend_volatility,recency_days,customer_tenure_days,active_months,monthly_spend_est,disposable_income,foir,savings_rate,dti,debt_per_card
0,CLTIND00001,25333.00,303996,25333,728,0,1,32,0.20,9593.16,2398.29,3098.96,1650.38,4,0.25,4,4,1.36,0.26,0.25,0.26,85,607,4,474.13,24858.87,0.02,0.98,0.00,0.00
1,CLTIND00002,59420.75,713049,59421,825,315841,5,19,0.56,45021.67,2813.85,5300.50,1119.11,16,0.62,16,11,2.67,0.47,0.25,0.47,171,852,16,1585.27,57835.48,0.03,0.97,0.44,63168.20
2,CLTIND00004,36796.58,441559,36797,757,121659,1,56,1.00,4319.10,4319.10,4319.10,4319.10,1,1.00,1,1,-0.00,NaN,0.00,NaN,606,0,1,4319.10,32477.48,0.12,0.88,0.28,121659.00


## 5. Feature Selection — Remove Redundant Features

In [10]:
from sklearn.feature_selection import VarianceThreshold

X = feature_store.drop(columns=['client_id','monthly_income','yearly_income'])
y = feature_store['monthly_income']

# Step 1: Drop high-correlation features
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]
X = X.drop(columns=to_drop)
print(f"Dropped {len(to_drop)} highly correlated features: {to_drop}")

# Step 2: Variance threshold (drop near-zero variance)
selector = VarianceThreshold(threshold=0.01)
X_selected = pd.DataFrame(selector.fit_transform(X), columns=X.columns[selector.get_support()])
print(f"After Variance Threshold: {X_selected.shape[1]} features remaining")

print("\nFinal feature list:")
for f in X_selected.columns:
    print(f"  - {f}")

Dropped 5 highly correlated features: ['merchant_diversity', 'city_diversity', 'active_months', 'disposable_income', 'savings_rate']
After Variance Threshold: 21 features remaining

Final feature list:
  - income_per_capita
  - credit_score
  - total_debt
  - num_credit_cards
  - age
  - tx_frequency_per_month
  - total_spend
  - avg_tx_amount
  - max_tx_amount
  - min_tx_amount
  - tx_count
  - chip_usage_ratio
  - spending_entropy
  - spending_volatility
  - weekend_spend_ratio
  - mom_spend_volatility
  - recency_days
  - customer_tenure_days
  - monthly_spend_est
  - dti
  - debt_per_card


## 6. Save Feature Store

In [11]:
os.makedirs('../retail_lending/data/processed', exist_ok=True)

feature_store_path = '../retail_lending/data/processed/feature_store.parquet'
feature_store.to_parquet(feature_store_path, index=False)
print(f"Feature store saved to: {feature_store_path}")
print(f"Shape: {feature_store.shape}")

Feature store saved to: ../retail_lending/data/processed/feature_store.parquet
Shape: (4941, 30)


## Summary

| Feature Family | Count |
|---|---|
| Income | 8 |
| Spending | 8 |
| Behaviour | 4 |
| Stability | 3 |
| Affordability | 6 |
| **Total** | **29** |

The feature store is ready. Proceed to **Milestone 2: Income Estimation Model**.